In [ ]:
#从各家公司偿付能力披露页面获取pdf_url


from selenium import webdriver
from webdriver_manager.microsoft import EdgeChromiumDriverManager
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import requests
import re
import time

def extract_elements_with_fallback(url, text_to_find, xpath):
    # 初始化WebDriver
    service = Service(r'C:\Users\Lu Quanlin\Python Workspace\CROSS II project\2025Q4\msedgedriver.exe')
    driver = webdriver.Edge(service=service)


    # 访问网页
    driver.get(url)
    time.sleep(3)

    link = None  # 初始化为None

    # 首先尝试按照特定文本筛选网页元素   绝大部分公司按关键词查找网页元素即可
    try:       
        element_by_text = driver.find_element(By.XPATH, f"//*[contains(text(), '{text_to_find}')]")
        if company in ['海保人寿']:
            element_by_text = element_by_text.find_element(By.XPATH, "./following-sibling::a")
        # 提取href属性
        link = element_by_text.get_attribute('href')
        if link == 'javascript:void(0)':
            link = element_by_text.get_attribute('boff')
        print(link)
    except Exception as e:
        print(f"Type1: 没有找到包含特定文本的元素，错误：{e}")

    # 首先尝试按照特定文本筛选网页元素   绝大部分公司按关键词查找网页元素即可
    try:       
        element_by_text = driver.find_element(By.XPATH, f"//a[contains(text(), '{text_to_find}')]")  #有的公司要指定查找a元素 比如信泰人寿
        if company in ['海保人寿']:
            element_by_text = element_by_text.find_element(By.XPATH, "./following-sibling::a")
        # 提取href属性
        link = element_by_text.get_attribute('href')
        if link == 'javascript:void(0)':
            link = element_by_text.get_attribute('boff')
        print(link)
    except Exception as e:
        print(f"Type1.5: 没有找到包含特定文本的元素，错误：{e}")

    # 如果没有找到，再按照特定文本筛选网页元素   剩下绝大部分公司需要找关键词所在元素的父类a元素
    if not link:
        try:
            element_by_text = driver.find_element(By.XPATH, f"//*[contains(text(), '{text_to_find}')]")
            current = element_by_text
            # 向上遍历父节点直到找到a元素或到达根节点
            while current:
                parent = current.find_element(By.XPATH, "..")
                if parent.tag_name == 'a':
                    print(f"Found parent a element: {parent.get_attribute('href')}")
                    link = parent.get_attribute('href')
                    break
                current = parent
            print(link)
        except Exception as e:
            print(f"Type2: 没有找到包含特定文本元素的父类a元素，错误：{e}")
            
    # 如果没有找到，再按照href链接内的关键词筛选网页元素
    if not link:
        try:
            element_by_href = driver.find_element(By.XPATH, f"//a[contains(@href, '{text_to_find}')]")
            link = element_by_href.get_attribute('href')
            print(link)
        except Exception as e:
            print(f"Type3: 没有找到包含特定文本的href链接，错误：{e}")
    
    # 如果没有找到，再按照xpath地址直接定位网页元素
    if not link:
        try:
            element_by_xpath = driver.find_element(By.XPATH, xpath)
            link = element_by_xpath.get_attribute('href')
            print(link)
        except Exception as e:
            print(f"Type4: 没有找到该xpath地址，错误：{e}")

    # 处理特定公司的逻辑
    if company in ['泰康养老']:  # 适用于泰康养老
        try:
            a_element = driver.find_element(By.XPATH, f"//a[contains(text(), '{text_to_find}')]")
            li_element = a_element.find_element(By.XPATH, "../..")
            onclick = li_element.get_attribute('onclick')
            match = re.search(r"window.open\('([^']+)'\)", onclick)
            if match:
                link = match.group(1)
                print(link)
        except Exception as e:
            print(f"处理泰康养老时出错：{e}")

    if company in ['瑞华健康', '招商仁和', '横琴人寿','陆家嘴国泰']:  
        try:
            key_text = text_to_find if company in ['瑞华健康', '横琴人寿','陆家嘴国泰'] else '偿付能力'
            element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '{key_text}')]"))
            )
            element.click()
            expanded_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '{text_to_find}')]"))
            )
            element_by_text = driver.find_element(By.XPATH, f"//a[contains(text(), '{text_to_find}')]")
            link = element_by_text.get_attribute('href')
            print(link)
        except Exception as e:
            print(f"处理瑞华健康时出错：{e}")

    if company in ['弘康人寿']:
        try:
            element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '专项信息')]"))
            )
            element.click()
            expanded_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '偿付能力')]"))
            )
            expanded_element.click()
            element_by_text = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '{text_to_find}')]"))
            )
            link = element_by_text.get_attribute('href')
            print(link)
        except Exception as e:
            print(f"处理弘康人寿时出错：{e}")

    if company in ['德华安顾']:
        try:
            element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '偿付能力')]"))
            )
            element.click()
            element_by_text = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//a[contains(@download, '{text_to_find}')]"))
            )
            link = element_by_text.get_attribute('href')
            print(link)
        except Exception as e:
            print(f"处理德华安顾时出错：{e}")

    if company in ['阳光人寿', '国民养老', '平安健康']:  # 等待新网页窗口出现并获取新网页窗口的url
        try:
            element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '{text_to_find}')]"))
            )
            element.click()
            WebDriverWait(driver, 10).until(EC.number_of_windows_to_be(2))
            windows = driver.window_handles
            driver.switch_to.window(windows[-1])
            link = driver.current_url
            print(link)
        except Exception as e:
            print(f"处理阳光人寿时出错：{e}")

    if company in ['中银三星']:
        try:
            element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '{text_to_find}')]"))
            )
            element.click()
            link = driver.current_url
            print(link)
        except Exception as e:
            print(f"处理中银三星时出错：{e}")

    # 关闭浏览器
    driver.quit()
    return link

def get_download_url(url, text_to_find, xpath):

    # 根据下载方式请求PDF文件
    #print(company + quarter)
    download_url = extract_elements_with_fallback(url, text_to_find, xpath)
    
    # 特殊处理国寿的下载链接
    if company in ['国华人寿','英大人寿','财信人寿','北京人寿','复星保德信','渤海人寿','信泰人寿'] and not download_url[0].endswith('.pdf'):
        suffix = extract_elements_with_fallback(download_url, text_to_find, xpath)
        download_url = construct_download_url(suffix)
        print(f"Constructed download URL: {download_url}")

    if company in ['中意人寿']:
        download_url = construct_download_url(download_url)
        print(f"Constructed download URL: {download_url}")

    if company in ['华汇人寿']:
        suffix = extract_elements_with_fallback(download_url, '见附件', xpath)
        download_url = construct_download_url(suffix)
        print(f"Constructed download URL: {download_url}")

    if company in ['中荷人寿']:
        suffix = extract_elements_with_fallback(download_url, '.pdf', xpath)
        download_url = construct_download_url(suffix)
        print(f"Constructed download URL: {download_url}")

    if company in ['人保养老']:
        driver = webdriver.Edge()
        driver.get(url)
        download_url = driver.current_url
        print(f"人保养老url: {download_url}")

    if company in ['阳光人寿']:
        file_index = download_url.find("file=")
        # 如果找到了file=，提取之后的部分
        if file_index != -1:
            download_url = download_url[file_index + 5:]

    if company in ['东吴人寿']:
        download_url = download_url.replace('/cs/pdfjs/web/viewer.html?', '')
        print('东吴人寿成功')
        
    if company in ['国寿养老']:
        download_url = download_url.replace('uiFramework/js/pdfjs/web/viewer.html?file=', '')
        print('国寿养老成功')        
        
    
    if not download_url:
        return 'not found'
    else:
        return download_url

def construct_download_url(suffix):
    if company in ['中国人寿']:
        base_url = 'https://www.e-chinalife.com/'
    if company in ['中意人寿']:
        base_url = 'https://www.generalichina.com/expandPDF/publish.html?PDF='
    else:
        base_url = ''
    return base_url + suffix

In [ ]:
#有了pdf_url后，下载pdf到指定路径并以指定文件名保存的方法

import requests
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
import os
import time

def try_download_with_requests(pdf_url, file_path, file_name):
    
    # 确保保存路径存在，如果不存在则创建
    if not os.path.exists(file_path):
        os.makedirs(file_path)
    
    # 完整的文件保存路径
    file_path = os.path.join(file_path, file_name)
    
    response = requests.get(pdf_url, stream=True)
    if response.status_code == 200 and (
        'application/pdf' in response.headers.get('Content-Type', '')):
        with open(file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("PDF downloaded successfully using requests.")
        return True

    if company in ['东吴人寿','国寿养老','新华养老','人保养老','大家养老','阳光人寿']:
        with open(file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("PDF downloaded successfully using requests.")
        return True 
        
    return False

def download_with_selenium(pdf_url, file_path, file_name):
    options = webdriver.EdgeOptions()
    # 设置下载路径为你希望保存文件的目录
    options.add_experimental_option("prefs", {
    "download.default_directory": file_path,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True
    })
    service = Service(r'C:\Users\Lu Quanlin\Python Workspace\CROSS II project\2025Q4\msedgedriver.exe')
    driver = webdriver.Edge(options=options,service=service)

    driver.get(pdf_url)
    time.sleep(5)
    if company == '华汇人寿':
        time.sleep(5)  # 等待文件下载完成
        # 获取最新下载的文件名
        old_file_name = max([file_path + "\\" + f for f in os.listdir(file_path)],key=os.path.getctime)
        # 修改文件名
        new_file_name = file_name
        os.rename(os.path.join(file_path, old_file_name), os.path.join(file_path, new_file_name))
        driver.quit()

    if company == '中意人寿':
        driver.switch_to.frame('PDF')

    if company == '中国人寿':
        driver.switch_to.frame('the-canvas')
        
    # 定位下载按钮并点击（根据实际情况修改选择器）
    try:
        #国寿，中意
        download_button = driver.find_element(By.ID, 'download')
    except Exception as e:
        print(f"没有找到id = download的元素，错误：{e}")
        download_button = None
        
    if download_button is None:
        try:
            download_button = driver.find_element(By.XPATH, f"//*[contains(text(), '{'点击下载'}')]") #Minsheng
        except Exception as e:
            print(f"没有找到包含'点击下载'的按钮，错误：{e}")
            download_button = None
            
    if download_button is None:
        try:
            download_button = driver.find_element(By.ID, 'save') #中信保诚
        except Exception as e:
            print(f"没有找到包含'点击下载'的按钮，错误：{e}")
            download_button = None

    if company == '中韩人寿':
        element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, 'download'))
        )
        element.click()
    
    if download_button is not None:
        download_button.click()  # 点击下载按钮
        time.sleep(5)  # 等待文件下载完成
        # 获取最新下载的文件名
        old_file_name = max([file_path + "\\" + f for f in os.listdir(file_path)],key=os.path.getctime)
        # 修改文件名
        new_file_name = file_name
        os.rename(os.path.join(file_path, old_file_name), os.path.join(file_path, new_file_name))
        return True
    else:
        print("无法找到下载按钮")
        return False

    driver.quit()

def download_pdf(pdf_url, file_path, file_name):   
    if company == '恒安标准养老':
        return download_with_selenium(pdf_url, file_path, file_name)
    if not try_download_with_requests(pdf_url, file_path, file_name):
        print("Attempting to download with Selenium.")
        return download_with_selenium(pdf_url, file_path, file_name)
    else:
        print("Direct download successful.")
        return True

In [ ]:
#从pdf把表格抽到excel

import pandas as pd
import camelot
import os

def extract_tables_to_excel(pdf_path, specific_chars):

    # 读取PDF文件中的表格
    tables = camelot.read_pdf(
        filepath=pdf_path,
        flavor='stream',
        pages='1-end',
        edge_tol=280,  # 如果漏了表格，把这个宽容度调高再试
        strip_text=' ',  # 去掉空格
        row_tol=15
    )

    # 提取PDF文件名（不含扩展名）
    pdf_filename = os.path.splitext(os.path.basename(pdf_path))[0]
    folder_path = os.path.dirname(pdf_path)
    
    # 创建Excel文件名
    output_excel_filename = f'{folder_path}\\{pdf_filename}_output.xlsx'

    # 写入Excel文件
    with pd.ExcelWriter(output_excel_filename) as writer:
        for i in range(1, len(tables)):
            # 检查DataFrame是否包含特定字符
            for specific_char in specific_chars:
                if specific_char in str(tables[i].df.values.flatten()) and ('数' in str(tables[i].df.values.flatten()) 
                                                                            or '末' in str(tables[i].df.values.flatten())):
                # 如果包含特定字符，则写入Excel
                    tables[i].df.to_excel(writer, sheet_name=f'{specific_char}_no{i}', index=False)
                else:
                    # 如果不包含特定字符，则跳过
                    print(f'Sheet {i} does not contain the specific character and is skipped.')

In [3]:
import fitz  # PyMuPDF
from openpyxl import Workbook
import os

def read_pdf_pages_with_keywords(pdf_path, keywords):
    """读取PDF中含有多个特定关键词的页面内容"""
    doc = fitz.open(pdf_path)
    pages_content = []
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text = page.get_text()
        if any(keyword in text for keyword in keywords):
            pages_content.append(text)
    doc.close()
    return '\n'.join(pages_content)  # 将所有匹配的页面内容合并为一个字符串

def write_to_excel(pdf_files, keywords, excel_path):
    """将含有多个特定关键词的PDF页面内容写入Excel，每个PDF一个单元格"""
    wb = Workbook()
    sheet = wb.active
    sheet.title = '重大投资&融资'
    
    # 设置表头
    sheet.cell(row=1, column=1).value = 'Company'
    sheet.cell(row=1, column=2).value = 'Content'
    
    for row_num, pdf_file in enumerate(pdf_files, start=2):
        content = read_pdf_pages_with_keywords(pdf_file, keywords)
        sheet.cell(row=row_num, column=1).value = pdf_file[len(directory_path)+1:pdf_file.rfind("20")]
        sheet.cell(row=row_num, column=2).value = content
    
    wb.save(excel_path)

# 示例用法
# pdf_files = ['file1.pdf', 'file2.pdf', 'file3.pdf']  # PDF文件列表
# keyword = '特定关键词'  # 需要查找的关键词
# excel_path = 'output.xlsx'  # 输出的Excel文件路径
# write_to_excel(pdf_files, keyword, excel_path)

In [1]:
import pdfplumber
import pandas as pd

#这一段是用pdfplumber取表格的方法
def extract_tables_from_pdf(pdf_path):
    """
    从PDF文件中提取表格。

    参数:
    pdf_path (str): PDF文件的路径。

    返回值:
    list: 包含所有提取的表格的列表，每个表格为一个pandas DataFrame。
    """
    tables = []  # 用于存储提取的表格
   
    with pdfplumber.open(pdf_path) as pdf:
        # 遍历每一页
        for page in pdf.pages:
            # 提取页面中的表格
            page_tables = page.extract_tables()
            for table in page_tables:
                # 将表格转换为DataFrame
                df = pd.DataFrame(table[1:], columns=table[0])
                tables.append(df)
    return tables


def is_not_blank(page):
    text = page.extract_text()
    return bool(text.strip())

def get_non_blank_page(pdf_file):
    # 使用 pdfplumber 检查每一页
    with pdfplumber.open(pdf_file) as pdf:
        non_blank_pages = [page.page_number for page in pdf.pages if is_not_blank(page)]

    # 将页码列表转换为字符串
    pages_to_read = ','.join(map(str, non_blank_pages))
    return pages_to_read


# 函数用于获取特定公司的参数，如果没有特定参数则返回默认参数
def get_company_params(company_name):
        #这一段是用camelot取表的参数
    default_params = {
        "flavor": "stream",
        "pages": "1-end",
        "edge_tol": 200,
        "strip_text": ' ',
        "row_tol": 12,
        "column_tol": 0.1
    }
    
    # 特定公司的参数
    company_specific_params = {
        "复星保德信": {
            "edge_tol": 40,
            "column_tol": 1,
            "row_tol": 12
        },
        "复星联合": {
            "edge_tol": 100,
            "column_tol": 0,
            "row_tol": 12
        },
        "国华人寿": {
            "edge_tol": 100,
            "column_tol": 10,
            "row_tol": 12
        },
        "瑞泰人寿": {
            "pages": get_non_blank_page(pdf_file)
        },
        "东吴人寿": {
            "edge_tol": 50,
            "column_tol": 0.1,
            "row_tol": 12
        },
        "弘康人寿": {
            "edge_tol": 50,
            "column_tol": 0.1,
            "row_tol": 12
        },
        "恒安标准养老": {
            "column_tol": 0.1
        },
        "国民养老": {
            "edge_tol": 100,
            "column_tol": 3
        },
        "中韩人寿": {
            "flavor": "lattice"
        }
    }
        
    company_params = company_specific_params.get(company_name, {})
    params = {**default_params, **company_params}
    
    # 如果是lattice模式，移除stream模式的参数
    if params["flavor"] == "lattice":
        for key in ["edge_tol", "row_tol", "column_tol"]:
            if key in params:
                del params[key]
                
    return params




def search_cell(df, include1_conditions, include2_conditions, exclude_conditions=None):
    """
    Search for a cell in df that matches the given include1 or include2 conditions and
    does not match the exclude conditions.
    """
    for i in range(len(df)):
        for j in range(len(df.columns)):
            # 检查是否包含 include1 中的全部关键词
            include1_match = all(include1_cond in str(df.iloc[i, j]) for include1_cond in include1_conditions)
            # 检查是否包含 include2 中的全部关键词
            if include2_conditions == []:
                include2_match = False
            else:
                include2_match = all(include2_cond in str(df.iloc[i, j]) for include2_cond in include2_conditions)
            # 检查是否不包含任何 exclude 的字符串
            exclude_match = exclude_conditions is None or not any(exclude_cond in str(df.iloc[i, j]) for exclude_cond in exclude_conditions)
            # 如果包含 include1 中的全部关键词或 include2 中的全部关键词，并且不包含 exclude 中的任何关键词
            if (include1_match or include2_match) and exclude_match:
                return i, j
    return None, None

def offset(df, keyphrase, rows=0, cols=0):
    """
    从df中搜索包含特定关键词，返回其往下rows行，往右cols列的内容。
    """
    # 定义关键词及其对应的搜索条件
    default_conditions_dict = {
        '认可资产': {
            'include1': ['认可资产'],
            'exclude': ['非认可资产', '其他认可资产','认可资产合计']
        },
        '认可负债': {
            'include1': ['认可负债'],
            'exclude': ['其他认可负债','认可负债合计']
        },
        '寿险业务保险风险最低资本合计': {
            'include1': ['寿险业务保险', '最低资本合计']
        },
        '非寿险业务保险风险最低资本合计': {
            'include1': ['非寿险业务保险', '最低资本合计']
        },
        '市场风险-最低资本合计': {
            'include1': ['市场风险', '最低资本合计']
        },
        '信用风险-最低资本合计': {
            'include1': ['信用风险', '最低资本合计']
        },
        '计入核心一级资本的保单未来盈余': {
            'include1': ['核心一级', '保单未来盈余']
        },
        '计入核心二级资本的保单未来盈余': {
            'include1': ['核心二级', '保单未来盈余']
        },
        '计入附属一级资本的保单未来盈余': {
            'include1': ['附属一级', '保单未来盈余']
        },
        '计入附属二级资本的保单未来盈余': {
            'include1': ['附属二级', '保单未来盈余']
        },
        '量化风险最低资本': {
            'include1': ['量化风险最低资本'],
            'include2': ['可资本化风险最低资本'],
            'exclude': ['未考虑特征系数前']
        },
        '量化风险最低资本（未考虑特征系数前）': {
            'include1': ['量化风险最低资本','未考虑特征系数前'],
            'include2': ['可资本化风险最低资本','未考虑特征系数前']
        },
        '量化风险分散效应': {
            'include1': ['量化风险分散效应'],
            'include2': ['可资本化风险分散效应']
        },
        '最低资本': {
            'include1': ['最低资本'],
            'exclude': ['风险最低资本']
        },
        '特定类别保险合同损失吸收效应': {
            'include1': ['特定类别保险合同损失吸收效应','损失吸收'],
            'include2': []
        },
        '寿险业务保险风险-损失发生风险最低资本': {
            'include1': ['寿险业务保险','损失发生'],
            'include2': []
        },
        '寿险业务保险风险-退保风险最低资本': {
            'include1': ['寿险业务保险','退保风险'],
            'include2': []
        },
        '寿险业务保险风险-费用风险最低资本': {
            'include1': ['寿险业务保险','费用风险'],
            'include2': []
        },
        '寿险业务保险风险-风险分散效应': {
            'include1': ['寿险业务保险','风险分散'],
            'include2': []
        },
        '非寿险业务保险风险-保费及准备金风险最低资本': {
            'include1': ['非寿险业务保险','保费及准备金风险'],
            'include2': []
        },
        '非寿险业务保险风险-巨灾风险最低资本': {
            'include1': ['非寿险业务保险','巨灾风险'],
            'include2': []
        },
        '非寿险业务保险风险-巨灾风险最低资本': {
            'include1': ['非寿险业务保险','风险分散'],
            'include2': []
        },
        '市场风险-利率风险最低资本': {
            'include1': ['市场风险','利率风险'],
            'include2': []
        },
        '市场风险-权益价格风险最低资本': {
            'include1': ['市场风险','权益价格风险'],
            'include2': []
        },
        '市场风险-房地产价格风险最低资本': {
            'include1': ['市场风险','房地产价格风险'],
            'include2': []
        },
        '市场风险-境外固定收益类资产价格风险最低资本': {
            'include1': ['市场风险','境外固定收益类资产'],
            'include2': []
        },
        '市场风险-境外权益类资产价格风险最低资本': {
            'include1': ['市场风险','境外权益类资产'],
            'include2': []
        },
        '市场风险-汇率风险最低资本': {
            'include1': ['市场风险','汇率风险'],
            'include2': []
        },
        '市场风险-风险分散效应': {
            'include1': ['市场风险','风险分散'],
            'include2': []
        },
        '信用风险-利差风险最低资本': {
            'include1': ['信用风险','利差风险'],
            'include2': []
        },
        '信用风险-交易对手违约风险最低资本': {
            'include1': ['信用风险','交易对手违约风险'],
            'include2': []
        },
        '信用风险-风险分散效应': {
            'include1': ['信用风险','风险分散'],
            'include2': []
        },
        '市场风险-风险分散效应': {
            'include1': ['市场风险','风险分散'],
            'include2': []
        },
        '市场风险-风险分散效应': {
            'include1': ['市场风险','风险分散'],
            'include2': []
        },
        '市场风险-风险分散效应': {
            'include1': ['市场风险','风险分散'],
            'include2': []
        },
        '损失吸收调整-不考虑上限': {
            'include1': ['损失吸收调整','不考虑上限'],
            'include2': []
        },
        '附加资本': {
            'include1': ['附加资本'],
            'exclude': ['附加资本','逆周期附加资本','D-SII附加资本','G-SII附加资本','其他附加资本']
        },
        '总资产': {
            'include1': ['总资产'],
            'exclude': ['收益率']
        },
        '净资产': {
            'include1': ['净资产'],
            'exclude': ['收益率']
        },
        '独立账户负债': {
            'include1': ['独立账户负债']
        },
        '保险业务收入': {
            'include1': ['保险业务收入']
        },
        '签单保费': {
            'include1': ['签单保费'],
            'exclude': ['续期签单保费','期交签单保费']
        },
        # ... 其他关键词条件
    }

    # 特定公司的特殊条件字典
    special_conditions_dict = {
        '中荷人寿': {
            '量化风险分散效应': {
                'include1': ['风险分散效应']
            },
            '特定类别保险合同损失吸收效应': {
                'include1': ['损失吸收']
            },
            '量化风险最低资本（未考虑特征系数前）': {
                'include1': ['量化风险最低资本','2.1+2.2+2.3-2.4-2.5']
            },
            '寿险业务保险风险最低资本合计': {
                'include1': ['寿险业务保险', '最低资本']
            },
            '非寿险业务保险风险最低资本合计': {
                'include1': ['非寿险业务保险', '最低资本']
            },
            '市场风险-最低资本合计': {
                'include1': ['市场风险', '最低资本']
            },
            '信用风险-最低资本合计': {
                'include1': ['信用风险', '最低资本']
            },
            '寿险业务保险风险-损失发生风险最低资本': {
                'include1': ['损失发生风险']
            },
            '寿险业务保险风险-退保风险最低资本': {
                'include1': ['退保风险最低资本']
            },
            '寿险业务保险风险-费用风险最低资本': {
                'include1': ['费用风险最低资本']
            },
            '寿险业务保险风险-风险分散效应': {
                'include1': ['寿险业务保险风险间的相关性效应'],
                'exclude': ['非寿险']
            },
            '非寿险业务保险风险-保费及准备金风险最低资本': {
                'include1': ['保费及准备金风险']
            },
            '非寿险业务保险风险-风险分散效应': {
                'include1': ['非寿险业务保险风险间的相关性效应']
            },
            '市场风险-利率风险最低资本': {
                'include1': ['利率风险最低资本']
            },
            '市场风险-权益价格风险最低资本': {
                'include1': ['权益价格风险最低资本']
            },
            '市场风险-权益价格风险最低资本': {
                'include1': ['权益价格风险最低资本']
            },
            '市场风险-房地产价格风险最低资本': {
                'include1': ['房地产价格风险最低资本']
            },
            '市场风险-境外固定收益类资产价格风险最低资本': {
                'include1': ['境外固收类资产价格风险最低资本']
            },
            '市场风险-境外权益类资产价格风险最低资本': {
                'include1': ['境外权益类资产价格风险最低资本']
            },
            '市场风险-汇率风险最低资本': {
                'include1': ['汇率风险最低资本']
            },
            '市场风险-风险分散效应': {
                'include1': ['市场风险间的相关性效应']
            },
            '信用风险-利差风险最低资本': {
                'include1': ['利差风险最低资本']
            },
            '信用风险-交易对手违约风险最低资本': {
                'include1': ['交易对手违约风险最低资本']
            },
            '信用风险-风险分散效应': {
                'include1': ['信用风险间的相关性效应']
            }
            # ... 公司A的其他特殊关键词条件
        },
        '公司B': {
            '认可负债': {
                'include1': ['公司B认可负债'],
                'exclude': ['公司B其他认可负债', '公司B认可负债合计']
            }
            # ... 公司B的其他特殊关键词条件
        }
        # ... 其他公司的特殊条件
    }

    # 确定使用的条件字典
    conditions_dict = default_conditions_dict.copy()  # 从默认字典开始

    if company_name and company_name in special_conditions_dict:
        company_specific_conditions = special_conditions_dict[company_name]
        for key, conditions in company_specific_conditions.items():
            if key == keyphrase:  # 只有当keyphrase匹配时才更新条件
                conditions_dict[key] = conditions  # 更新条件

    # 检查df是否为空
    if df.empty:
        return "DataFrame is empty"

    # 获取搜索条件
    conditions = conditions_dict.get(keyphrase, {'include1': [keyphrase], 'include2': [], 'exclude': []})
    include1_conditions = conditions.get('include1', [])
    include2_conditions = conditions.get('include2', [])
    exclude_conditions = conditions.get('exclude', None)

    # 搜索单元格
    row_index, col_index = search_cell(df, include1_conditions, include2_conditions, exclude_conditions)

    # 检查是否有匹配的行
    if row_index is not None:
        try:
            # 返回偏移后的单元格内容
            return df.iloc[row_index + rows, col_index + cols]
        except IndexError:
            # 如果索引超出范围，返回错误信息
            return "Index out of range"
    else:
        return None

# 示例使用
# 假设df是你的DataFrame，keyphrase是你要搜索的关键词
# result = offset(df, keyphrase)
# print(result)